# Constant-alpha price drift

Load the stored no-agent benchmark, inspect its summary, and plot the 10-second mid-price response on both linear and log–log axes. The notebook uses only the Python standard library plus IPython's built-in HTML display.

In [4]:
from pathlib import Path
import json

candidates = [
    Path('constant_alpha_drift.json'),
    Path('benchmarks/queue_reactive/constant_alpha_drift.json'),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not locate constant_alpha_drift.json')

with data_path.open() as file:
    benchmark = json.load(file)

print(f'Data: {data_path.resolve()}')
print(f"Constant alpha: {benchmark['constant_alpha']:+g}")
print(f"Horizon: {benchmark['horizon_seconds']:g} seconds")
print(f"Sequences per scale: {benchmark['agents_per_scale']:,}")
print(f"Agent action: {benchmark['agent_action']}")
print(f"Power-law impact: {benchmark['power_law_impact']}")

Data: /Users/josephleclere/Documents/PufferLib/benchmarks/queue_reactive/constant_alpha_drift.json
Constant alpha: +1
Horizon: 10 seconds
Sequences per scale: 4,096
Agent action: Noop
Power-law impact: False


In [5]:
from html import escape
from IPython.display import HTML, display

results = benchmark['results']
columns = [
    ('Scale', 'alpha_scale', 'g'),
    ('Mean drift', 'mean_drift', '.4f'),
    ('Std. error', 'standard_error', '.4f'),
    ('5th pct.', 'q05', '.2f'),
    ('Median', 'median', '.2f'),
    ('95th pct.', 'q95', '.2f'),
    ('Sequences', 'n', ',d'),
]
header = ''.join(f'<th style="text-align:right;padding:5px 9px">{escape(label)}</th>' for label, _, _ in columns)
body = []
for row in results:
    cells = ''.join(
        f'<td style="text-align:right;padding:5px 9px;border-top:1px solid #9994">{row[key]:{fmt}}</td>'
        for _, key, fmt in columns
    )
    body.append(f'<tr>{cells}</tr>')
display(HTML(
    '<table style="border-collapse:collapse;font-variant-numeric:tabular-nums">'
    f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'
))

Scale,Mean drift,Std. error,5th pct.,Median,95th pct.,Sequences
0,0.0488,0.0485,-5.00,0.00,5.00,"4,096"
0.03125,0.1398,0.0487,-5.00,0.00,5.00,"4,096"
0.0625,0.2588,0.0489,-5.00,0.00,5.50,"4,096"
0.125,0.5229,0.0493,-5.00,0.50,6.00,"4,096"
0.25,0.9095,0.0515,-4.50,1.00,6.50,"4,096"
0.5,1.8201,0.0543,-4.00,2.00,8.00,"4,096"
1,4.8881,0.0645,-1.00,5.00,12.00,"4,096"
2,15.6713,0.0957,6.00,15.50,26.00,"4,096"
4,37.6901,0.2355,13.50,37.50,63.00,"4,096"
8,44.8688,0.4339,4.50,43.50,86.00,"4,096"


In [6]:
import math

width, height = 820, 440
left, right, top, bottom = 72, 24, 28, 62
plot_width = width - left - right
plot_height = height - top - bottom
positive_scales = [row['alpha_scale'] for row in results if row['alpha_scale'] > 0]
log_min, log_max = math.log2(min(positive_scales)), math.log2(max(positive_scales))
baseline_gap = 76

def x_position(scale):
    if scale == 0:
        return left
    fraction = (math.log2(scale) - log_min) / (log_max - log_min)
    return left + baseline_gap + fraction * (plot_width - baseline_gap)

all_y = [0.0]
for row in results:
    all_y.extend([row['q05'], row['q95']])
y_min, y_max = min(all_y), max(all_y)
padding = 0.08 * (y_max - y_min)
y_min, y_max = y_min - padding, y_max + padding

def y_position(value):
    fraction = (value - y_min) / (y_max - y_min)
    return top + plot_height * (1 - fraction)

grid_values = list(range(-20, 101, 20))
grid = []
for value in grid_values:
    y = y_position(value)
    grid.append(f'<line x1="{left}" x2="{width-right}" y1="{y:.2f}" y2="{y:.2f}" stroke="currentColor" opacity="0.12"/>')
    grid.append(f'<text x="{left-10}" y="{y+4:.2f}" text-anchor="end">{value}</text>')

upper = [(x_position(row['alpha_scale']), y_position(row['q95'])) for row in results]
lower = [(x_position(row['alpha_scale']), y_position(row['q05'])) for row in reversed(results)]
band_points = ' '.join(f'{x:.2f},{y:.2f}' for x, y in upper + lower)
mean_points = ' '.join(
    f"{x_position(row['alpha_scale']):.2f},{y_position(row['mean_drift']):.2f}"
    for row in results
)
marks = []
for row in results:
    x, y = x_position(row['alpha_scale']), y_position(row['mean_drift'])
    ci_low = y_position(row['mean_drift'] - 1.96 * row['standard_error'])
    ci_high = y_position(row['mean_drift'] + 1.96 * row['standard_error'])
    marks.append(f'<line x1="{x:.2f}" x2="{x:.2f}" y1="{ci_low:.2f}" y2="{ci_high:.2f}" stroke="#268cff"/>')
    marks.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="4" fill="#268cff"><title>scale {row["alpha_scale"]:g}: {row["mean_drift"]:.4f} ± {1.96*row["standard_error"]:.4f} ticks</title></circle>')

x_ticks = []
for row in results:
    scale = row['alpha_scale']
    x = x_position(scale)
    x_ticks.append(f'<text x="{x:.2f}" y="{height-bottom+24}" text-anchor="middle">{scale:g}</text>')

svg = f'''
<svg viewBox="0 0 {width} {height}" style="width:100%;max-width:{width}px;color:inherit;font:12px system-ui" role="img" aria-label="Constant alpha price drift curve">
  {''.join(grid)}
  <line x1="{left}" x2="{width-right}" y1="{y_position(0):.2f}" y2="{y_position(0):.2f}" stroke="currentColor" opacity="0.55"/>
  <polygon points="{band_points}" fill="#268cff" opacity="0.14"/>
  <polyline points="{mean_points}" fill="none" stroke="#268cff" stroke-width="2.5"/>
  {''.join(marks)}
  {''.join(x_ticks)}
  <text x="{left + plot_width/2}" y="{height-12}" text-anchor="middle" font-weight="600">Alpha intensity scale (log₂ spacing after zero)</text>
  <text x="18" y="{top + plot_height/2}" text-anchor="middle" font-weight="600" transform="rotate(-90 18 {top + plot_height/2})">Mid-price drift after 10 s (ticks)</text>
</svg>
'''
display(HTML(svg))

In [7]:
# Log-log view. Scale zero is omitted because log(0) is undefined.
log_results = [row for row in results if row['alpha_scale'] > 0 and row['mean_drift'] > 0]
log_x = [math.log(row['alpha_scale']) for row in log_results]
log_y = [math.log(row['mean_drift']) for row in log_results]
x_bar, y_bar = sum(log_x) / len(log_x), sum(log_y) / len(log_y)
log_slope = sum((x-x_bar)*(y-y_bar) for x, y in zip(log_x, log_y)) / sum((x-x_bar)**2 for x in log_x)
log_intercept = y_bar - log_slope*x_bar
residual_ss = sum((y-(log_intercept + log_slope*x))**2 for x, y in zip(log_x, log_y))
total_ss = sum((y-y_bar)**2 for y in log_y)
log_r_squared = 1 - residual_ss / total_ss
top_pair_slope = math.log(log_results[-1]['mean_drift'] / log_results[-2]['mean_drift']) / math.log(2)
print(f'Global log-log fit: drift ∝ scale^{log_slope:.3f}  (R²={log_r_squared:.4f})')
print(f'Local slope from scale 4 to 8: {top_pair_slope:.3f}  (saturation would push this toward zero)')

width, height = 820, 440
left, right, top, bottom = 72, 24, 34, 62
plot_width = width - left - right
plot_height = height - top - bottom
x_min, x_max = min(log_x), max(log_x)
ci_lower = [row['mean_drift'] - 1.96*row['standard_error'] for row in log_results]
ci_upper = [row['mean_drift'] + 1.96*row['standard_error'] for row in log_results]
y_min, y_max = math.log(min(ci_lower)), math.log(max(ci_upper))
y_pad = 0.08 * (y_max-y_min)
y_min, y_max = y_min-y_pad, y_max+y_pad

def log_x_position(scale):
    return left + (math.log(scale)-x_min) / (x_max-x_min) * plot_width

def log_y_position(value):
    return top + (1-(math.log(value)-y_min)/(y_max-y_min)) * plot_height

y_tick_candidates = [0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
y_ticks = [value for value in y_tick_candidates if math.exp(y_min) <= value <= math.exp(y_max)]
grid = []
for value in y_ticks:
    y = log_y_position(value)
    grid.append(f'<line x1="{left}" x2="{width-right}" y1="{y:.2f}" y2="{y:.2f}" stroke="currentColor" opacity="0.12"/>')
    grid.append(f'<text x="{left-10}" y="{y+4:.2f}" text-anchor="end">{value:g}</text>')

mean_points = ' '.join(
    f"{log_x_position(row['alpha_scale']):.2f},{log_y_position(row['mean_drift']):.2f}"
    for row in log_results
)
scale_one_mean = next(row['mean_drift'] for row in log_results if row['alpha_scale'] == 1)
linear_reference = ' '.join(
    f"{log_x_position(row['alpha_scale']):.2f},{log_y_position(scale_one_mean*row['alpha_scale']):.2f}"
    for row in log_results
)
marks = []
for row in log_results:
    x, y = log_x_position(row['alpha_scale']), log_y_position(row['mean_drift'])
    ci_low = log_y_position(row['mean_drift'] - 1.96*row['standard_error'])
    ci_high = log_y_position(row['mean_drift'] + 1.96*row['standard_error'])
    marks.append(f'<line x1="{x:.2f}" x2="{x:.2f}" y1="{ci_low:.2f}" y2="{ci_high:.2f}" stroke="#268cff"/>')
    marks.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="4" fill="#268cff"><title>scale {row["alpha_scale"]:g}: {row["mean_drift"]:.4f} ticks</title></circle>')

x_ticks = []
for row in log_results:
    scale = row['alpha_scale']
    x = log_x_position(scale)
    x_ticks.append(f'<text x="{x:.2f}" y="{height-bottom+24}" text-anchor="middle">{scale:g}</text>')

log_svg = f'''
<div style="font:14px system-ui;margin-top:20px"><strong>Log–log view</strong></div>
<svg viewBox="0 0 {width} {height}" style="width:100%;max-width:{width}px;color:inherit;font:12px system-ui" role="img" aria-label="Log-log constant alpha price drift curve">
  {''.join(grid)}
  <polyline points="{linear_reference}" fill="none" stroke="#df8b2e" stroke-width="1.5" stroke-dasharray="6 5"/>
  <polyline points="{mean_points}" fill="none" stroke="#268cff" stroke-width="2.5"/>
  {''.join(marks)}
  {''.join(x_ticks)}
  <text x="{width-right-4}" y="{top+16}" text-anchor="end" fill="#df8b2e">dashed: exactly proportional, anchored at scale 1</text>
  <text x="{left + plot_width/2}" y="{height-12}" text-anchor="middle" font-weight="600">Alpha intensity scale (log)</text>
  <text x="18" y="{top + plot_height/2}" text-anchor="middle" font-weight="600" transform="rotate(-90 18 {top + plot_height/2})">Mean mid-price drift after 10 s (log ticks)</text>
</svg>
'''
display(HTML(log_svg))

Global log-log fit: drift ∝ scale^1.118  (R²=0.9865)
Local slope from scale 4 to 8: 0.252  (saturation would push this toward zero)


The JSON also contains `sequence_drifts` for every scale, so distributional or alternative confidence-interval analyses can be added without rerunning the simulator.